# RNN Encoder-Decoder for Statistical Machine Translation




Theory

A Recurrent Neural Network (RNN) is a neural network designed to process sequential data such as text. It maintains a hidden state that carries information from previous words, enabling it to learn relationships within a sequence. However, standard RNNs have difficulty capturing long-term dependencies due to the vanishing gradient problem.

The Sequence-to-Sequence (Seq2Seq) model is an encoder–decoder architecture commonly used for machine translation. The encoder reads the input sentence and converts it into a context vector that summarizes its meaning. The decoder uses this context vector to generate the translated sentence one word at a time until an end-of-sequence (EOS) token is produced.

Before training, sentences are preprocessed by converting them to lowercase, removing unnecessary characters, tokenizing words, and mapping them to numerical indices using a vocabulary. Special SOS (Start of Sequence) and EOS (End of Sequence) tokens are added to each sentence.

During training, teacher forcing is used, where the decoder receives the correct target word as input at each step instead of its own prediction. This speeds up convergence and improves training stability. The model is trained using Negative Log Likelihood Loss (NLLLoss) and backpropagation to minimize prediction errors.

After training, the model translates new input sentences by encoding them into a context vector and decoding the output sequence word by word. Encoder–decoder RNNs form the basis of many natural language processing applications, including machine translation, text summarization, chatbots, and speech recognition.

**Requirements**

In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data



- To make a word list from sentence, let's create a Class the sentence/line and creates `word2index`, `index2word`, and `word2count`.

In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

- The input the the `addSentence` method that is required to make the `word2index`, `index2word`, and `word2count` is (obviously) a sentence. 
- Therefore, so let's create a method to 
    - read the Dataset/file, 
    - split it into lines and then 
    - create sentence pairs (Language1 Sentence, Equivalent Language2 Sentence) 
    - Normalize.

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

Since there are a lot of example sentences and we want to train something quickly, we’ll trim the data set to only relatively short and simple sentences. Here the maximum length is 10 words (that includes ending punctuation) and we’re filtering to sentences that translate to the form “I am” or “He is” etc. (accounting for apostrophes replaced earlier).

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.

In [5]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Now, let's bundle everything up as per the diagram above: 

In [6]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [8]:
PATH = r'data\eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes tres solitaire', 'you re very lonely']


15

`Note`: Here, we have only inserted lower case in the word2index, therefore, use of uppercase letters will yield an error. 

### Encoder

In [9]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

Here, we convert the input word indices into dense embedding vectors. During training, these embeddings are learned and gradually capture the semantic relationships and meanings of the words.


### Decoder

In [10]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

## Training



## Prepare Training Data

For each pair:
-  we will need an input tensor (indexes of the words in the input sentence) and 
- target tensor (indexes of the words in the target sentence). 
- While creating these vectors we will append the EOS token to both sequences.

In [11]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [12]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [13]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [14]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [15]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [16]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [17]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [18]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 4s (- 2m 51s) (5 2%) 1.9787
0m 6s (- 2m 8s) (10 5%) 1.2929
0m 9s (- 1m 51s) (15 7%) 1.1096
0m 11s (- 1m 43s) (20 10%) 0.9884
0m 13s (- 1m 37s) (25 12%) 0.8847
0m 16s (- 1m 32s) (30 15%) 0.7923
0m 18s (- 1m 27s) (35 17%) 0.7097
0m 21s (- 1m 25s) (40 20%) 0.6338
0m 24s (- 1m 24s) (45 22%) 0.5678
0m 27s (- 1m 22s) (50 25%) 0.5082
0m 30s (- 1m 19s) (55 27%) 0.4509
0m 33s (- 1m 17s) (60 30%) 0.3985
0m 36s (- 1m 15s) (65 32%) 0.3582
0m 39s (- 1m 12s) (70 35%) 0.3186
0m 42s (- 1m 10s) (75 37%) 0.2855
0m 45s (- 1m 7s) (80 40%) 0.2543
0m 47s (- 1m 4s) (85 42%) 0.2297
0m 50s (- 1m 2s) (90 45%) 0.2039
0m 54s (- 0m 59s) (95 47%) 0.1856
0m 57s (- 0m 57s) (100 50%) 0.1688
1m 0s (- 0m 54s) (105 52%) 0.1538
1m 3s (- 0m 51s) (110 55%) 0.1409
1m 6s (- 0m 49s) (115 57%) 0.1265
1m 10s (- 0m 46s) (120 60%) 0.1210
1m 13s (- 0m 44s) (125 62%) 0.1133
1m 17s (- 0m 41s) (130 65%) 0.103

In [19]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> vous etes tres bons
= you re very good
< you re very good <EOS>

> je vais bien
= i m well
< they re all nuts <EOS>

> vous etes une puritaine
= you re a prude
< you re a prude <EOS>

> tu es tres perspicace
= you re very perceptive
< you re very perceptive <EOS>

> vous etes beaux
= you are beautiful
< he is watching tv <EOS>

> il est intelligent
= he is intelligent
< i m very cold <EOS>

> je suis encore marie
= i m still married
< i m still married <EOS>

> tu es avide
= you re greedy
< we are making progress <EOS>

> il est extremement beau
= he is tremendously handsome
< he is tremendously handsome <EOS>

> elles sont presque la
= they re almost here
< they re almost here <EOS>



`Note`: Could train this with varying degree of success. 

Discussion

In this experiment, an RNN Encoder-Decoder network has been employed to translate sentences from one language to another through the use of a sequence-to-sequence method. In particular, the encoder network is able to learn the mapping between the input sentence and the context vector, whereas the decoder creates the output sentence, translating the input word-by-word. The model is trained in such a way that the loss function consistently decreases, showing that the model learns to find relations between the source and target languages. Teacher forcing helps to speed up training. Although the translation results were satisfactory for short sentences, the performance can deteriorate in case of long sentences because of limitations of basic RNNs.

Conclusion

The experiment was conducted to demonstrate the use of an RNN Encoder-Decoder Model in Machine Translation through Sequence-to-Sequence Architecture. The task involved learning of translation through the encoding and decoding of sentences. Some of the factors which were found to be important in enhancing the process include data preprocessing, word embedding, and teacher forcing. The lab session gave a good insight into neural machine translation and the working of encoder-decoder models.